# Capstone 2 — Telecom Customer Churn Prediction
### Microsoft Fabric Medallion + Data Science Capstone

**Business scenario:** "ConnectMobile" wants to predict which subscribers are likely to churn next month so the retention team can proactively reach out.

**Fabric capability highlighted:** Lakehouse Medallion architecture feeding a **Data Science** workflow — feature engineering in Gold, model training tracked with **MLflow**, and a scored output table ready for Power BI.

**What this notebook builds:**
1. Synthetic subscriber, usage, billing, and support-ticket data
2. Bronze → Silver → Gold (feature mart: `gold.mart_churn_features`)
3. A churn classification model trained and logged with MLflow
4. A scored output table `gold.mart_churn_predictions` for the retention team

Attach this notebook to a Lakehouse (e.g. `telecom_capstone_lakehouse`) before running.

In [ ]:
%pip install faker --quiet

In [ ]:
import random
from datetime import datetime, timedelta
import pandas as pd
from pyspark.sql import functions as F
from faker import Faker

fake = Faker()
Faker.seed(11)
random.seed(11)

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

NUM_CUSTOMERS = 6000

## 1. Generate source data

In [ ]:
plans = ["Basic", "Standard", "Premium", "Unlimited"]
contract_types = ["Month-to-month", "1 Year", "2 Year"]

customers = []
for i in range(1, NUM_CUSTOMERS + 1):
    tenure_months = random.randint(1, 72)
    plan = random.choice(plans)
    contract = random.choices(contract_types, weights=[0.55, 0.25, 0.20])[0]
    monthly_charge = {"Basic": 25, "Standard": 45, "Premium": 70, "Unlimited": 95}[plan] + random.uniform(-5, 5)

    # Build in realistic churn drivers: month-to-month + short tenure + high support tickets -> more likely to churn
    support_tickets = max(0, int(random.gauss(2, 2)))
    base_churn_prob = 0.05
    if contract == "Month-to-month":
        base_churn_prob += 0.20
    if tenure_months < 6:
        base_churn_prob += 0.15
    if support_tickets >= 4:
        base_churn_prob += 0.20
    churned = random.random() < min(base_churn_prob, 0.9)

    customers.append({
        "customer_id": f"SUB{i:06d}",
        "age": random.randint(18, 80),
        "plan": plan,
        "contract_type": contract,
        "tenure_months": tenure_months,
        "monthly_charge": round(monthly_charge, 2),
        "support_tickets_90d": support_tickets,
        "autopay_enabled": random.random() > 0.4,
        "churned": churned,
        "signup_date": (datetime.utcnow() - timedelta(days=30 * tenure_months)).date().isoformat(),
    })

df_customers = pd.DataFrame(customers)
print(f"Generated {len(df_customers):,} subscribers | churn rate: {df_customers['churned'].mean():.1%}")

In [ ]:
# Monthly usage for the last 3 months per customer
usage = []
for _, row in df_customers.iterrows():
    for m in range(3):
        usage.append({
            "customer_id": row["customer_id"],
            "usage_month": m + 1,
            "data_gb": round(max(0, random.gauss(8, 4)), 2),
            "voice_minutes": max(0, int(random.gauss(200, 100))),
            "sms_count": max(0, int(random.gauss(50, 40))),
        })
df_usage = pd.DataFrame(usage)
print(f"Generated {len(df_usage):,} monthly usage rows")

## 2. Bronze layer

In [ ]:
def to_bronze(pdf, table_name):
    sdf = spark.createDataFrame(pdf).withColumn("_ingestion_timestamp", F.current_timestamp())
    sdf.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table_name)
    print(f"{table_name}: {sdf.count():,} rows")

to_bronze(df_customers, "bronze.customers")
to_bronze(df_usage, "bronze.usage")

## 3. Silver layer — clean and conform

In [ ]:
silver_customers = (spark.table("bronze.customers")
    .dropDuplicates(["customer_id"])
    .withColumn("signup_date", F.to_date("signup_date"))
    .withColumn("monthly_charge", F.col("monthly_charge").cast("decimal(10,2)")))
silver_customers.write.format("delta").mode("overwrite").saveAsTable("silver.customers")

silver_usage = spark.table("bronze.usage").dropDuplicates(["customer_id", "usage_month"])
silver_usage.write.format("delta").mode("overwrite").saveAsTable("silver.usage")

print(f"silver.customers: {silver_customers.count():,} | silver.usage: {silver_usage.count():,}")

## 4. Gold layer — churn feature mart

In [ ]:
usage_agg = (spark.table("silver.usage")
    .groupBy("customer_id")
    .agg(F.avg("data_gb").alias("avg_data_gb"),
         F.avg("voice_minutes").alias("avg_voice_minutes"),
         F.avg("sms_count").alias("avg_sms_count")))

mart_churn_features = (spark.table("silver.customers")
    .join(usage_agg, "customer_id", "left")
    .select("customer_id", "age", "plan", "contract_type", "tenure_months", "monthly_charge",
            "support_tickets_90d", "autopay_enabled", "avg_data_gb", "avg_voice_minutes",
            "avg_sms_count", "churned"))

mart_churn_features.write.format("delta").mode("overwrite").saveAsTable("gold.mart_churn_features")
print(f"gold.mart_churn_features: {mart_churn_features.count():,} rows")
display(mart_churn_features.limit(5))

## 5. Train a churn model, tracked with MLflow
This uses scikit-learn for simplicity; the same pattern applies to SynapseML/MLlib for larger data volumes.

In [ ]:
%pip install scikit-learn mlflow --quiet

In [ ]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import OneHotEncoder
import numpy as np

pdf = mart_churn_features.toPandas()

feature_cols_num = ["age", "tenure_months", "monthly_charge", "support_tickets_90d",
                     "avg_data_gb", "avg_voice_minutes", "avg_sms_count"]
feature_cols_cat = ["plan", "contract_type", "autopay_enabled"]

X_cat = pd.get_dummies(pdf[feature_cols_cat].astype(str), drop_first=True)
X = pd.concat([pdf[feature_cols_num].fillna(0), X_cat], axis=1)
y = pdf["churned"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

mlflow.set_experiment("telecom_churn_capstone")
with mlflow.start_run(run_name="random_forest_baseline"):
    model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight="balanced")
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)

    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 8)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("roc_auc", auc)
    mlflow.sklearn.log_model(model, "churn_model")

    print(f"Accuracy: {acc:.3f} | ROC-AUC: {auc:.3f}")
    print(classification_report(y_test, preds))

## 6. Score all customers and write the prediction mart

In [ ]:
pdf["churn_probability"] = model.predict_proba(X)[:, 1]
pdf["risk_band"] = pd.cut(pdf["churn_probability"], [0, 0.3, 0.6, 1.0], labels=["Low", "Medium", "High"])

scored_sdf = spark.createDataFrame(
    pdf[["customer_id", "plan", "contract_type", "tenure_months", "churn_probability", "risk_band"]]
    .assign(risk_band=lambda d: d["risk_band"].astype(str))
)
scored_sdf.write.format("delta").mode("overwrite").saveAsTable("gold.mart_churn_predictions")

print(f"gold.mart_churn_predictions: {scored_sdf.count():,} rows")
display(scored_sdf.groupBy("risk_band").count().orderBy("risk_band"))
print("\nCapstone 2 (Telecom Customer Churn) complete.")